# ICT4403 Basic Transformer English French
Kaggle run: https://www.kaggle.com/code/eltonlewis/ict4403-basic-transformer-english-french (version 3).
This notebook was assembled from the Kaggle MCP source and execution log. The code cell is the script executed by Kaggle; stream outputs are the saved run log. Checkpoints and data files are separate Kaggle output artifacts.
Saved output files: basic_transformer/metrics.json, basic_transformer/model.pt, basic_transformer/predictions.jsonl, basic_transformer/shared_bpe.model, basic_transformer/shared_bpe.vocab, basic_transformer/tokenizer_corpus.txt, data/processed/manifest.json, data/processed/test.jsonl, data/processed/train.jsonl, data/processed/validation.jsonl.


In [1]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q","datasets>=3.0","sentencepiece>=0.2"])
"""Deterministic English-to-French OPUS Books preprocessing."""
import argparse
import hashlib
import json
import random
import re
import unicodedata
from pathlib import Path

DATASET = "Helsinki-NLP/opus_books"
CONFIG = "en-fr"
REVISION = "main"


def clean(value):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", value or "")).strip()


def prepare(rows, train_size=4000, val_size=200, test_size=200, seed=4403):
    """Deduplicate on source, then shuffle; source strings cannot cross splits."""
    unique = {}
    rejected = {"empty_or_length": 0, "length_ratio": 0, "duplicate_source": 0}
    for row in rows:
        pair = row["translation"]
        en, fr = clean(pair["en"]), clean(pair["fr"])
        if not (10 <= len(en) <= 300 and 10 <= len(fr) <= 350):
            rejected["empty_or_length"] += 1
            continue
        ratio = len(fr) / len(en)
        if not 0.45 <= ratio <= 2.2:
            rejected["length_ratio"] += 1
            continue
        key = en.casefold()
        if key in unique:
            rejected["duplicate_source"] += 1
            continue
        uid = hashlib.sha256((en + "\0" + fr).encode()).hexdigest()[:16]
        unique[key] = {"id": uid, "en": en, "fr": fr}
    pairs = list(unique.values())
    random.Random(seed).shuffle(pairs)
    n = train_size + val_size + test_size
    if len(pairs) < n:
        raise ValueError(f"Need {n} clean unique pairs; found {len(pairs)}")
    picked = pairs[:n]
    splits = {
        "train": picked[:train_size],
        "validation": picked[train_size:train_size + val_size],
        "test": picked[train_size + val_size:],
    }
    return splits, {"raw_rows": len(rows), "clean_unique_rows": len(pairs),
                    "rejected": rejected, "seed": seed,
                    "sizes": {k: len(v) for k, v in splits.items()},
                    "source": f"https://huggingface.co/datasets/{DATASET}/tree/{REVISION}/{CONFIG}",
                    "language_pair": "en-fr", "normalization": "Unicode NFKC and whitespace collapse",
                    "filters": "English 10-300 chars, French 10-350 chars, length ratio 0.45-2.2",
                    "deduplication": "casefolded English source; first occurrence retained"}


def write_splits(splits, manifest, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    for name, items in splits.items():
        with (output / f"{name}.jsonl").open("w", encoding="utf-8") as f:
            for item in items:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
    (output / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--output", default="data/processed")
    p.add_argument("--train-size", type=int, default=4000)
    p.add_argument("--val-size", type=int, default=200)
    p.add_argument("--test-size", type=int, default=200)
    p.add_argument("--seed", type=int, default=4403)
    a = p.parse_args()
    from datasets import load_dataset
    rows = load_dataset(DATASET, CONFIG, split="train", revision=REVISION)
    splits, manifest = prepare(rows, a.train_size, a.val_size, a.test_size, a.seed)
    write_splits(splits, manifest, a.output)
    print(json.dumps(manifest, indent=2))



sys.argv=['prepare_data']
main()
"""Small Transformer trained from scratch on the shared OPUS Books split."""
import argparse
import json
import math
import time
from pathlib import Path

import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import DataLoader


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]


class BasicTransformer(nn.Module):
    def __init__(self, vocab, d_model=256, nhead=4, layers=2, ff=512, max_len=160):
        super().__init__()
        self.token = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.net = nn.Transformer(d_model=d_model, nhead=nhead,
                                  num_encoder_layers=layers, num_decoder_layers=layers,
                                  dim_feedforward=ff, dropout=0.1, batch_first=True)
        self.out = nn.Linear(d_model, vocab)
        self.scale = math.sqrt(d_model)

    def embed(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.token(x) * self.scale + self.pos(pos)

    def forward(self, source, target):
        mask = nn.Transformer.generate_square_subsequent_mask(target.size(1), device=target.device)
        hidden = self.net(self.embed(source), self.embed(target), tgt_mask=mask,
                          src_key_padding_mask=source.eq(0),
                          tgt_key_padding_mask=target.eq(0),
                          memory_key_padding_mask=source.eq(0))
        return self.out(hidden)


def batchify(items, sp, device, max_len=96):
    def encode(text):
        return [sp.bos_id()] + sp.encode(text)[:max_len - 2] + [sp.eos_id()]
    en = [encode(r["en"]) for r in items]
    fr = [encode(r["fr"]) for r in items]
    def pad(seqs):
        width = max(map(len, seqs))
        return torch.tensor([s + [0] * (width - len(s)) for s in seqs], device=device)
    return pad(en), pad(fr)


@torch.inference_mode()
def translate(model, sp, sources, device, max_len=96):
    src, _ = batchify([{"en": x, "fr": x} for x in sources], sp, device, max_len)
    target = torch.full((len(sources), 1), sp.bos_id(), dtype=torch.long, device=device)
    done = torch.zeros(len(sources), dtype=torch.bool, device=device)
    for _ in range(max_len - 1):
        next_token = model(src, target)[:, -1].argmax(-1)
        next_token = torch.where(done, sp.eos_id(), next_token)
        target = torch.cat([target, next_token[:, None]], dim=1)
        done |= next_token.eq(sp.eos_id())
        if bool(done.all()):
            break
    return [sp.decode([t for t in row[1:].tolist() if t not in (0, sp.eos_id())])
            for row in target]


def run(data_dir, output, epochs=5, batch_size=64, seed=4403):
    torch.manual_seed(seed)
    if not torch.cuda.is_available():
        raise RuntimeError("This full experiment requires a CUDA GPU")
    device = torch.device("cuda")
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    train = read_jsonl(Path(data_dir) / "train.jsonl")
    val = read_jsonl(Path(data_dir) / "validation.jsonl")
    test = read_jsonl(Path(data_dir) / "test.jsonl")
    corpus = output / "tokenizer_corpus.txt"
    corpus.write_text("\n".join(x[k] for x in train for k in ("en", "fr")), encoding="utf-8")
    prefix = str(output / "shared_bpe")
    spm.SentencePieceTrainer.train(input=str(corpus), model_prefix=prefix,
        vocab_size=4000, model_type="bpe", character_coverage=1.0,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3, hard_vocab_limit=False)
    sp = spm.SentencePieceProcessor(model_file=prefix + ".model")
    model = BasicTransformer(sp.vocab_size()).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scaler = torch.amp.GradScaler("cuda")
    history = []
    start = time.perf_counter()
    best_loss = float("inf")
    for epoch in range(epochs):
        model.train()
        losses = []
        loader = DataLoader(train, batch_size=batch_size, shuffle=True, collate_fn=lambda z: z)
        for items in loader:
            src, tgt = batchify(items, sp, device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(src, tgt[:, :-1])
                loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                                    tgt[:, 1:].reshape(-1), ignore_index=0)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        model.eval()
        val_losses = []
        with torch.inference_mode():
            for i in range(0, len(val), batch_size):
                src, tgt = batchify(val[i:i + batch_size], sp, device)
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(src, tgt[:, :-1])
                    loss = nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                                        tgt[:, 1:].reshape(-1), ignore_index=0)
                val_losses.append(loss.item())
        row = {"epoch": epoch + 1, "train_loss": sum(losses) / len(losses),
               "validation_loss": sum(val_losses) / len(val_losses)}
        history.append(row)
        print(row, flush=True)
        if row["validation_loss"] < best_loss:
            best_loss = row["validation_loss"]
            torch.save(model.state_dict(), output / "model.pt")
    training_seconds = time.perf_counter() - start
    model.load_state_dict(torch.load(output / "model.pt", map_location=device, weights_only=True))
    model.eval()
    torch.cuda.synchronize()
    start = time.perf_counter()
    predictions = []
    for i in range(0, len(test), 16):
        predictions += translate(model, sp, [r["en"] for r in test[i:i + 16]], device)
    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - start
    with (output / "predictions.jsonl").open("w", encoding="utf-8") as f:
        for row, prediction in zip(test, predictions):
            f.write(json.dumps({**row, "prediction": prediction}, ensure_ascii=False) + "\n")
    metrics = {"model": "basic_transformer", "architecture": "encoder-decoder",
               "parameters": sum(p.numel() for p in model.parameters()),
               "training_seconds": training_seconds, "inference_seconds": inference_seconds,
               "test_count": len(test), "epochs": epochs, "history": history,
               "decoding": "greedy, max 95 generated tokens", "device": torch.cuda.get_device_name(0)}
    (output / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    return metrics



print(json.dumps(run('data/processed', '/kaggle/working/basic_transformer'), indent=2), flush=True)

Generating train split: 100%|██████████| 127085/127085 [00:00<00:00, 500966.27 examples/s]


{


  "raw_rows": 127085,


  "clean_unique_rows": 117720,


  "rejected": {


    "empty_or_length": 7990,


... (325 outputs left out)

/usr/local/lib/python3.13/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'


  text = re.sub(r'_', '\_', text) # Escape underscores in display text


[NbConvertApp] Converting notebook __script__.ipynb to html


[NbConvertApp] Writing 334014 bytes to __results__.html
